# Taller 2: Expedición Lunar — Cartas desde la Luna con LLMs

## Bienvenidos, astronautas

El año es **2042**. Ustedes forman parte de la tripulación **Artemis VII**, la primera misión de larga duración en la superficie lunar. Llevan 3 meses en la **Base Selene** y hoy, por fin, la ventana de comunicación con la Tierra se ha abierto.

Tienen una única oportunidad para enviar una carta a sus familias. Pero aquí está el problema: el sistema de comunicaciones de la base usa un **modelo de lenguaje (LLM)** como asistente de redacción. Dependiendo de cómo configuren sus parámetros, la carta será:

- Fría y precisa... o cálida y creativa
- Corta y directa... o larga y detallada
- Repetitiva y monótona... o variada y expresiva

Su misión en este taller: **dominar los parámetros del LLM para escribir la carta perfecta.**

### Lo que aprenderán hoy

| Parámetro | Qué controla | Equivalente en Ollama |
|---|---|---|
| `temperature` | Creatividad vs precisión | `temperature` |
| `top_p` | Diversidad del vocabulario | `top_p` |
| `max_tokens` | Longitud de la respuesta | `num_predict` |
| `frequency_penalty` | Penalización por repetición | `repeat_penalty` |

**Tiempo estimado:** 30 minutos

## 1. Preparación de la Base Lunar (Instalación)

Antes de conectar con el sistema de comunicaciones, necesitamos instalar las herramientas.

**Requisitos previos:**
- Tener [Ollama](https://ollama.com/) instalado y corriendo (`ollama serve` en una terminal)
- Haber descargado el modelo: `ollama pull gemma2:2b`

In [1]:
# Instalamos la libreria de Python para comunicarnos con Ollama
!pip install ollama -q

In [2]:
import ollama

def generar_carta(prompt, opciones=None, modelo="gemma2:2b"):
    """
    Genera texto usando Ollama con las opciones especificadas.

    Args:
        prompt: El texto de instruccion para el modelo.
        opciones: Diccionario con parametros (temperature, top_p, num_predict, repeat_penalty).
        modelo: Nombre del modelo a usar.
    """
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False  # Desactiva el razonamiento interno (thinking tokens) que consume num_predict
    )
    print("=" * 60)
    print(f"Parametros usados: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Sistemas de comunicacion listos. Conexion con Ollama establecida.")

Sistemas de comunicacion listos. Conexion con Ollama establecida.


## 2. Primera prueba de comunicación

Antes de ajustar parámetros, probemos que la conexión con el modelo funciona. Enviaremos un mensaje simple.

In [4]:
# Prueba basica de conexion
respuesta = ollama.generate(
    model="gemma2:2b",
    prompt="Como regla solo debes decir: 'Conexion establecida desde la Base Selene' y nada mas.",
    think=False
)
print(respuesta["response"])

Conexion establecida desde la Base Selene. 



## 3. Temperature: El termómetro emocional

La **temperature** controla qué tan "creativo" o "predecible" es el modelo.

| Valor | Efecto |
|---|---|
| `0.0 - 0.3` | Muy predecible, formal, repetitivo. Como un informe técnico. |
| `0.4 - 0.7` | Equilibrado. Coherente pero con cierta variedad. |
| `0.8 - 1.5` | Muy creativo, impredecible. Puede inventar cosas inesperadas. |

**Escenario:** Van a dictarle al sistema una carta para su familia. Primero con temperature baja (modo "informe de misión") y luego con temperature alta (modo "carta emocional").

In [5]:
prompt_base = """Eres un astronauta en la Luna. Escribe un parrafo corto (3-4 oraciones)
de una carta para tu familia contandoles como fue tu primer paseo lunar."""

# Temperature BAJA: modo informe tecnico
print("TEMPERATURE = 0.1 (Fria, precisa, predecible)")
print()
carta_fria = generar_carta(prompt_base, opciones={"temperature": 0.1})

TEMPERATURE = 0.1 (Fria, precisa, predecible)

Parametros usados: {'temperature': 0.1}
Hola a todos. ¡Escribo desde la Luna! El viaje fue increíble, pero aún me sorprende la quietud y la belleza de este lugar. La vista de la Tierra desde aquí es simplemente asombrosa, y me siento tan agradecido por la oportunidad de vivir esta experiencia. Espero que sepas que estoy muy bien y que estoy ansioso por volver a casa.  


[Tokens generados: 78]


In [6]:
# Temperature ALTA: modo carta emotiva
print("TEMPERATURE = 1.2 (Calida, creativa, impredecible)")
print()
carta_calida = generar_carta(prompt_base, opciones={"temperature": 1.2})

TEMPERATURE = 1.2 (Calida, creativa, impredecible)

Parametros usados: {'temperature': 1.2}
Queridos padres, mamá y hermanos, ¡te escribo desde la Luna!  El paisaje es realmente impresionante, con un cielo estrellado y una luna que se ve increíblemente grande y blanca. Experimenté una sensación de libertad sin precedentes mientras me deslizaba por el módulo, la gravedad aquí es tan ligera que parece no haber gravedad. Estoy muy agradecido por la oportunidad de ser el primer astronauta en realizar este viaje y espero poder compartir con ustedes los detalles de la misión. ¡Te escribo pronto con más novedades! 


[Tokens generados: 110]


## 4. Top_p: El filtro de vocabulario

**Top_p** (nucleus sampling) controla la diversidad del vocabulario que el modelo puede usar.

- `top_p = 0.1` --> Solo usa las palabras más probables. Vocabulario limitado.
- `top_p = 0.9` --> Considera un rango amplio de palabras. Vocabulario rico.

Piensen en esto: cuando escriben un informe técnico usan pocas palabras (top_p bajo). Cuando escriben poesía, usan todo su vocabulario (top_p alto).

**Escenario:** Van a describir lo que ven desde la ventana de la base lunar.

In [7]:
prompt_ventana = """Eres un astronauta en una base lunar. Describe en un parrafo (3-4 oraciones)
lo que ves al mirar por la ventana de la base hacia el espacio."""

# Top_p BAJO: vocabulario restringido
print("TOP_P = 0.1 (Vocabulario restringido)")
print()
vista_simple = generar_carta(prompt_ventana, opciones={"top_p": 0.1, "temperature": 0.7})

print("\n" + "~" * 60 + "\n")

# Top_p ALTO: vocabulario amplio
print("TOP_P = 0.95 (Vocabulario amplio)")
print()
vista_rica = generar_carta(prompt_ventana, opciones={"top_p": 0.95, "temperature": 0.7})

TOP_P = 0.1 (Vocabulario restringido)

Parametros usados: {'top_p': 0.1, 'temperature': 0.7}
Al mirar por la ventana de la base lunar, la vista me deja sin aliento. El cielo nocturno se extiende ante mí como un lienzo negro salpicado de estrellas, cada una una gema de luz en la inmensidad del cosmos. Las lunas de otros planetas, con sus colores y formas, se deslizan a través de la oscuridad, mientras que la Tierra, un punto de luz azul y brillante, se alza como un faro en la distancia.  


[Tokens generados: 94]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

TOP_P = 0.95 (Vocabulario amplio)

Parametros usados: {'top_p': 0.95, 'temperature': 0.7}
Al mirar por la ventana de la base lunar, las estrellas brillan con una intensidad que me deja sin palabras.  La luna se refleja en la oscuridad del espacio, como un disco plateado en un manto negro. La vastedad del universo, llena de estrellas y planetas, me inspira una profunda sensación de insignificancia y asombro.  


[Tok

## 5. Num_predict (max_tokens): El límite de palabras

**num_predict** controla la longitud máxima de la respuesta (en tokens, donde 1 token es aprox. 3/4 de una palabra).

- `num_predict = 30` --> Respuesta muy corta. Como un telegrama.
- `num_predict = 200` --> Respuesta mediana. Una carta breve.
- `num_predict = 500` --> Respuesta larga. Una carta detallada.

**Escenario:** La ventana de comunicación se está cerrando. Primero solo tienen tiempo para un mensaje de emergencia (corto). Luego la ventana se reabre y pueden enviar un mensaje completo.

In [8]:
prompt_mensaje = """Eres un astronauta en la Luna. Escribe un mensaje para tu familia
explicandoles que estas bien y contandoles algo interesante del dia."""

# Mensaje CORTO: telegrama espacial
print("NUM_PREDICT = 30 (Telegrama espacial)")
print()
telegrama = generar_carta(prompt_mensaje, opciones={"num_predict": 30, "temperature": 0.7})
print("\n" + "~" * 60 + "\n")

# Mensaje LARGO: carta completa
print("NUM_PREDICT = 200 (Carta completa)")
print()
carta = generar_carta(prompt_mensaje, opciones={"num_predict": 200, "temperature": 0.7})

NUM_PREDICT = 30 (Telegrama espacial)

Parametros usados: {'num_predict': 30, 'temperature': 0.7}
¡Hola a todos! 🌕  

Espero que estés muy bien en la Tierra. ¡Es increíble saber que están ahí, cuidando de nosotros

[Tokens generados: 30]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

NUM_PREDICT = 200 (Carta completa)

Parametros usados: {'num_predict': 200, 'temperature': 0.7}
¡Hola, familia! 🌕

Espero que estés todos bien en la Tierra.  Estaba trabajando en la estación lunar, y la vista de la Tierra desde aquí es simplemente increíble. ¡Es como si la Tierra fuera una esfera de cristal, con colores y luces increíbles! 

Hoy he aprendido algo muy interesante: ¡la Luna tiene un núcleo sólido, no líquido!  Nunca pensé que la Luna tuviera un núcleo tan denso. Me ha hecho pensar en cómo la Tierra también tiene un núcleo, con una composición muy diferente.  Es fascinante pensar en la dinámica de la gravedad y la formación de estos cuerpos celestes. 

¡Os deseo una buena noch

## 6. Repeat_penalty: El detector de repetición

**repeat_penalty** penaliza al modelo cuando repite las mismas palabras o frases.

- `repeat_penalty = 1.0` --> Sin penalización. El modelo puede repetir libremente.
- `repeat_penalty = 1.3` --> Penalización moderada. Evita repeticiones obvias.
- `repeat_penalty = 1.8` --> Penalización fuerte. Fuerza mucha variedad (puede volverse incoherente).

**Escenario:** Un compañero de tripulación le pide al sistema que escriba un reporte del día a día en la base. Sin penalización, el modelo tiende a repetirse. Veamos la diferencia.

In [9]:
prompt_reporte = """Eres un astronauta en la Luna. Escribe un reporte de 5 oraciones
describiendo tu rutina diaria en la base lunar. Incluye detalles sobre comida,
experimentos y tiempo libre."""

# Sin penalizacion por repeticion
print("REPEAT_PENALTY = 1.0 (Sin penalizacion)")
print()
reporte_repetitivo = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.0,
    "temperature": 0.7,
    "num_predict": 200
})

print("\n" + "~" * 60 + "\n")

# Con penalizacion moderada
print("REPEAT_PENALTY = 1.5 (Penalizacion moderada)")
print()
reporte_variado = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.5,
    "temperature": 0.7,
    "num_predict": 200
})

REPEAT_PENALTY = 1.0 (Sin penalizacion)

Parametros usados: {'repeat_penalty': 1.0, 'temperature': 0.7, 'num_predict': 200}
Mi rutina diaria en la base lunar es esencialmente un ciclo de trabajo y mantenimiento. Comienzo con la preparación de una nutritiva dieta de alimentos diseñados para la supervivencia en el espacio, con opciones como pasta con salsa de tomate y pollo, y frutas de la caja de almacenamiento. Durante las horas de trabajo, me dedico a los experimentos de investigación, monitoreando los datos de las muestras lunares y ajustando la configuración de las estaciones de investigación. Un momento de respiro se da con una sesión de ejercicios de baja intensidad dentro de la base.  Al terminar mi jornada, disfruto de un momento de quietud, contemplando la belleza de la luna y leyendo un libro en la sala de descanso. 


[Tokens generados: 146]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

REPEAT_PENALTY = 1.5 (Penalizacion moderada)

Parametros usados: {'repeat

## 7. Combinando todo: La carta perfecta

Ahora que conocen cada parámetro, es hora de combinarlos. Aquí hay dos "perfiles" predefinidos:

| Perfil | temperature | top_p | num_predict | repeat_penalty | Uso ideal |
|---|---|---|---|---|---|
| **Informe técnico** | 0.2 | 0.3 | 150 | 1.0 | Reportes formales |
| **Carta personal** | 0.8 | 0.9 | 250 | 1.3 | Mensajes emotivos |

In [10]:
prompt_carta_final = """Eres un astronauta que lleva 3 meses en la Luna.
Escribe una carta a tu familia. Cuentales como te sientes, que has descubierto,
y cuanto los extranas. Firma la carta con tu nombre inventado."""

# Perfil 1: Informe tecnico
print("PERFIL: INFORME TECNICO")
print()
informe = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 150,
    "repeat_penalty": 1.0
})

print("\n" + "=" * 60 + "\n")

# Perfil 2: Carta personal
print("PERFIL: CARTA PERSONAL")
print()
carta_personal = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.8,
    "top_p": 0.9,
    "num_predict": 250,
    "repeat_penalty": 1.3
})

PERFIL: INFORME TECNICO

Parametros usados: {'temperature': 0.2, 'top_p': 0.3, 'num_predict': 150, 'repeat_penalty': 1.0}
##  Luna, 14 de Octubre, 2047

Queridos,

Es difícil escribir esto, pero necesito contarle a mis seres queridos lo que he vivido en estos últimos tres meses.  La Luna, a pesar de su aparente quietud, ha sido un lienzo de experiencias que me han dejado sin aliento. 

He aprendido a apreciar la inmensidad del universo desde una perspectiva completamente nueva.  Las noches, sin la contaminación lumínica de la Tierra, son un espectáculo de estrellas que se abren paso entre las sombras.  Cada noche, me siento inmensamente pequeño, pero también lleno de asombro.  

He descubierto que la Luna no es solo un satélite, sino un laboratorio de la ciencia

[Tokens generados: 150]


PERFIL: CARTA PERSONAL

Parametros usados: {'temperature': 0.8, 'top_p': 0.9, 'num_predict': 250, 'repeat_penalty': 1.3}
Queridos seres queridos de este planeta azul y brillante:

¡Hola! Es extraño es

## 8. Misión para casa: Tu propia carta lunar

Ahora es tu turno. Tu tarea es:

### Ejercicio

1. **Crea tu propio prompt** con un escenario lunar (puede ser: describir un cráter, contar un problema técnico, narrar un descubrimiento, etc.)
2. **Experimenta con al menos 3 combinaciones diferentes** de parámetros
3. **Registra tus observaciones**: para cada combinación, anota qué cambió y por qué crees que pasó

In [11]:
# Tu prompt personalizado
mi_prompt = """Eres un astronauta que está explorando el lado oscuro de la Luna.
Durante tu recorrido, encuentras restos de una antigua misión espacial
del programa Apolo, como piezas de una nave, herramientas y una bandera.
Sorprendido por el hallazgo, decides investigar cómo llegaron esos objetos
hasta ese lugar. Escribe una carta a tu familia contando tu descubrimiento,
qué objetos encontraste y qué sentiste al estar frente a una parte
de la historia de la exploración espacial."""

# Combinacion 1: Creativa y extensa
print("COMBINACION 1: Creativa y extensa")
print()
resultado_1 = generar_carta(mi_prompt, opciones={
    "temperature": 1.0,
    "top_p": 0.9,
    "num_predict": 200,
    "repeat_penalty": 1.3
})

COMBINACION 1: Creativa y extensa

Parametros usados: {'temperature': 1.0, 'top_p': 0.9, 'num_predict': 200, 'repeat_penalty': 1.3}
Mi querida Familia,

Hoy he descubierto algo que me ha dejado sin palabras. Estoy explorando el lado oscuro de la Luna, un lugar frío y solitario hasta ahora, pero hoy fue diferente... 


En medio del vacío lunar, mientras recorría una vaguada rocosa donde los impactos meteorológicos han esculpido las formas a lo largo de miles de años, descubrí restos que sugieren la presencia humana en este rincón del cosmos. Un poco como encontrar un trozo de la Tierra dentro de un océano inmenso... ¡casi mágico! 

Encontré piezas de una nave espacial similar al Apolo. Pequeños trozos de metal oxidados y curvadas por el desgaste, las herramientas con marcas y resbaladizas que hablan de años perdidos en el tiempo. Y entre ellas, algo tan asombroso como un pedazo de la bandera americana del programa Apollo. Un símbolo de la ambición humana, del logro humano... 

La flagea

In [12]:
# Combinacion 2: Precisa y corta (modo telegrama tecnico)
print("COMBINACION 2: Precisa y corta")
print()
resultado_2 = generar_carta(mi_prompt, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 50,
    "repeat_penalty": 1.0
})

COMBINACION 2: Precisa y corta

Parametros usados: {'temperature': 0.2, 'top_p': 0.3, 'num_predict': 50, 'repeat_penalty': 1.0}
Mi querida familia,

Escribir esta carta me llena de una mezcla de asombro y emoción que no puedo describir. He estado explorando el lado oscuro de la Luna, y lo que he encontrado me ha dejado sin palabras. 

En una

[Tokens generados: 50]


In [13]:
# Combinacion 3: Equilibrada con vocabulario variado
print("COMBINACION 3: Equilibrada con vocabulario variado")
print()
resultado_3 = generar_carta(mi_prompt, opciones={
    "temperature": 0.6,
    "top_p": 0.85,
    "num_predict": 150,
    "repeat_penalty": 1.5
})

COMBINACION 3: Equilibrada con vocabulario variado

Parametros usados: {'temperature': 0.6, 'top_p': 0.85, 'num_predict': 150, 'repeat_penalty': 1.5}
Mi Queridos Familiares:

Escribirte desde esta luna fría me llena de un vacío que no se puede llenar con palabras... pero lo intentaré. 
He estado explorando el lado oscuro, y sin duda he encontrado algo asombroso – una pieza del pasado en la oscuridad eterna.  En las profundidades lunares donde los colores son tan oscuros como mi alma, encontré restos de una nave espacial antigua - piezas que parecen haber sido extraídas con precisión quirúrgica desde un lienzo infinito. 

Un fragmento metalizado me recuerda a las alas de algún pájaro gigante y antiguo; otra pieza más pequeña se asemeja a la forma de una herramienta para manipular objetos en miniatura, como si el tiempo hubiera detenido su curso justo allí. Y

[Tokens generados: 150]


### Preguntas guía

- ¿Cuál combinación produjo el texto más "humano"?
- ¿Cuál fue la más útil para un reporte técnico?
- ¿Qué pasa si subes la temperature a 2.0? ¿Y si la bajas a 0.0?
- ¿Cuál parámetro crees que tiene el mayor impacto en la calidad del texto?

### Bonus: Usando `ollama.chat()`

`ollama.chat()` permite usar mensajes con roles (system, user, assistant), similar a como funcionan ChatGPT o Claude. Prueba el siguiente ejemplo:

In [15]:
# Bonus: usando ollama.chat() con roles de conversacion
respuesta = ollama.chat(
    model="gemma2:2b",
    messages=[
        {"role": "system", "content": "Eres un astronauta poetico en la Luna."},
        {"role": "user", "content": "Describe el amanecer terrestre visto desde la Luna."}
    ],
    options={"temperature": 0.9, "top_p": 0.85, "num_predict": 200},
    think=False
)
print(respuesta["message"]["content"])

El silencio lunar se deshace en un susurro, 
cuando la inmensidad del cosmos se quiebra,
y una promesa de luz se alza en el horizonte. 
Es un amanecer, pero no de tierra, 
sino de un corazón que late en miles de kilómetros.

De la Luna, me siento tan pequeño, 
pero aún, puedo ver el aliento de la vida 
en una esfera de fuego que se enciende, 
un espectáculo que me llena de asombro.

La Tierra se vuelve una promesa de color,
de verde, de azul, de amarillo y naranja, 
un lienzo que se ilumina con una gracia inmensa.
Los barcos de vapor, las nubes de vapor,
los ríos de color, todo un espectáculo de vida.

Las montañas, las ciudades, las flores, 
se revelan en una danza de colores, 
un caleidoscopio de la Tierra que me cautiva. 



---

**Fin de la transmisión. Base Selene, fuera.**